# Libraries & GPU Setup

In [ ]:
import os
import cv2 
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence 
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras import mixed_precision 
import matplotlib.pyplot as plt
import math 
import pandas as pd
import sys

# Check GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"Mixed Precision Policy: {policy.compute_dtype}")

Num GPUs Available:  1
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA RTX A4000, compute capability 8.6
Mixed Precision Policy: float16


# Directories & Hyperparameters

In [ ]:
# Data Paths
DATA_PATH = r'C:\Users\makvanmt1\Downloads\Thesis Data\Training Data'
OUTPUT_DIR = os.path.join(DATA_PATH, 'saved_models_v5')

# Image Specs
IMG_CHANNELS = 1 
PATCH_SIZE = 96  
PATCH_STRIDE = 48 

# Training Hyperparameters
EPOCHS = 5000 
BATCH_SIZE = 1 
LEARNING_RATE = 1e-4 
STEPS_PER_EPOCH = 5000

# The generator will randomly apply degradations up to these limits
MAX_NOISE_LEVEL = 0.05 
MAX_BLUR_SIGMA = 1.5   

# Model Architecture & Loss 

In [ ]:
def build_har_cnn(input_shape):
    
    #  Helper: Channel Attention Block 
    def channel_attention(x, filters, ratio=8):
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Reshape((1, 1, filters))(se)
        se = layers.Dense(filters // ratio, activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
        se = layers.Dense(filters, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)
        return layers.Multiply()([x, se])

    #  Helper: Residual Block 
    def res_block(x, filters):
        shortcut = x
        
        x = layers.Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    
        x = layers.Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        
        x = channel_attention(x, filters)
        x = layers.Add()([shortcut, x])
        x = layers.Activation('relu')(x)
        return x

    #  Main Architecture 
    inputs = layers.Input(input_shape)
    
    # 1. Feature Extraction
    x = layers.Conv2D(64, (9, 9), padding='same', kernel_initializer='he_normal', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # 2. Deep Body (8 Blocks)
    for _ in range(8):
        x = res_block(x, 64)
    
    # 3. Post-Residual Mapping
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_initializer='he_normal', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # 4. Bottleneck
    x = layers.Conv2D(32, (1, 1), padding='same', kernel_initializer='he_normal', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # 5. Reconstruction (Predict Noise)
    residual = layers.Conv2D(IMG_CHANNELS, (5, 5), padding='same', kernel_initializer='he_normal', dtype='float32')(x)

    # 6. Output Clean Image
    outputs = layers.Add()([inputs, residual])
    model = Model(inputs=[inputs], outputs=[outputs])
    
    # --- Loss & Metrics ---
    def ssim_loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    
    def composite_loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        alpha = 0.84
        return alpha * ssim_loss(y_true, y_pred) + (1 - alpha) * tf.reduce_mean(tf.abs(y_true - y_pred))
    
    def ssim_metric(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        return tf.image.ssim(y_true, y_pred, max_val=1.0)
    
    def psnr_metric(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        return tf.image.psnr(y_true, y_pred, max_val=1.0)

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE), 
        loss=composite_loss, 
        metrics=[ssim_metric, psnr_metric] 
    )
    
    return model

# Data Generator

In [ ]:
class RamDataGenerator(Sequence):
    def __init__(self, data_path, filenames, batch_size=1, 
                 n_channels=1, patch_size=96, stride=48, 
                 shuffle=True, steps_per_epoch=None, 
                 augment=False, robust=False):
        
        self.data_path = data_path
        self.input_img_path = os.path.join(self.data_path, "input")
        self.target_img_path = os.path.join(self.data_path, "target")
        self.filenames = filenames
        self.batch_size = batch_size
        self.n_channels = n_channels
        self.patch_size = patch_size
        self.stride = stride
        self.shuffle = shuffle
        self.steps_per_epoch = steps_per_epoch
        self.augment = augment 
        self.robust = robust 
        self.offset = 0 
        
        self.loaded_inputs = []
        self.loaded_targets = []
        
        self.__preload_all_data()
        self.indexes = np.arange(len(self.loaded_inputs))
        
        if self.shuffle:
            np.random.shuffle(self.indexes)
            
    def __preload_all_data(self):
        print(f"--- Pre-loading {len(self.filenames)} images... ---")
        for idx, fname in enumerate(self.filenames):
            img_path = os.path.join(self.input_img_path, fname)
            target_path = os.path.join(self.target_img_path, fname)
            
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            target = cv2.imread(target_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None or target is None: continue
            
            if img.shape != target.shape:
                target = cv2.resize(target, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_AREA)
            
            if img.shape[0] < self.patch_size or img.shape[1] < self.patch_size:
                continue

            img = np.expand_dims(img, axis=-1)
            target = np.expand_dims(target, axis=-1)
            
            self.loaded_inputs.append(img)
            self.loaded_targets.append(target)
            
            if len(self.loaded_inputs) % 5000 == 0:
                print(f"Loaded {len(self.loaded_inputs)}...")

        print(f"--- Loaded {len(self.loaded_inputs)} valid pairs. ---")

    def __len__(self):
        if self.steps_per_epoch:
            return self.steps_per_epoch
        return int(np.ceil(len(self.loaded_inputs) / self.batch_size))

    def __getitem__(self, index):
        real_index = (self.offset + index) % len(self.indexes)
        
        X_full = self.loaded_inputs[self.indexes[real_index]]
        y_full = self.loaded_targets[self.indexes[real_index]]
        
        X_batch = (X_full.astype('float32') / 255.0)
        y_batch = (y_full.astype('float32') / 255.0)
        
        X_batch = np.expand_dims(X_batch, axis=0)
        y_batch = np.expand_dims(y_batch, axis=0)
        
        x_out, y_out = self.__create_patches(X_batch, y_batch)
        
        if self.augment:
            x_out, y_out = self.__augment_batch(x_out, y_out)
            
        if self.robust:
            x_out = self.__degrade_batch(x_out)
            
        return x_out, y_out

    def __augment_batch(self, x, y):
        if np.random.rand() > 0.5: 
            x = np.flip(x, axis=2)
            y = np.flip(y, axis=2)
        if np.random.rand() > 0.5: 
            x = np.flip(x, axis=1)
            y = np.flip(y, axis=1)
        k = np.random.randint(0, 4) 
        if k > 0:
            x = np.rot90(x, k, axes=(1, 2))
            y = np.rot90(y, k, axes=(1, 2))
        return x, y

    def __degrade_batch(self, x):
        batch_size = x.shape[0]
        degraded_x = x.copy()
        
        for i in range(batch_size):
            if np.random.rand() > 0.5:
                sigma = np.random.uniform(0.1, MAX_BLUR_SIGMA)
                img = degraded_x[i, :, :, 0]
                img = cv2.GaussianBlur(img, (0, 0), sigma)
                degraded_x[i, :, :, 0] = img

            if np.random.rand() > 0.5:
                noise_level = np.random.uniform(0, MAX_NOISE_LEVEL)
                noise = np.random.normal(0, noise_level, degraded_x[i].shape)
                degraded_x[i] = degraded_x[i] + noise
                
        return np.clip(degraded_x, 0.0, 1.0)

    def on_epoch_end(self):
        if self.steps_per_epoch:
            self.offset += self.steps_per_epoch
            if self.offset >= len(self.indexes):
                self.offset = 0
                if self.shuffle: np.random.shuffle(self.indexes)
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __create_patches(self, X_batch, y_batch):
        X_tensor = tf.convert_to_tensor(X_batch, dtype=tf.float32)
        y_tensor = tf.convert_to_tensor(y_batch, dtype=tf.float32)
        ksize = [1, self.patch_size, self.patch_size, 1]
        strides = [1, self.stride, self.stride, 1]
        rates = [1, 1, 1, 1]
        
        X_patches = tf.image.extract_patches(X_tensor, ksize, strides, rates, 'VALID')
        y_patches = tf.image.extract_patches(y_tensor, ksize, strides, rates, 'VALID')
        
        X_patches = tf.reshape(X_patches, [-1, self.patch_size, self.patch_size, self.n_channels])
        y_patches = tf.reshape(y_patches, [-1, self.patch_size, self.patch_size, self.n_channels])
        return X_patches.numpy(), y_patches.numpy()

# Utility Functions & Plotting

In [ ]:
def get_filenames(data_path):
    input_img_path = os.path.join(data_path, "input")
    target_img_path = os.path.join(data_path, "target")
    if not os.path.isdir(input_img_path): return []
    return sorted(list(set(os.listdir(input_img_path)) & set(os.listdir(target_img_path))))

def plot_history(history):
    plt.figure(figsize=(12, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss (Robust Blind Restoration)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_loss.png'))
    plt.show() 
  
    
    if 'ssim_metric' in history.history:
        plt.figure(figsize=(12, 6))
        plt.plot(history.history['ssim_metric'], label='Training SSIM')
        plt.plot(history.history['val_ssim_metric'], label='Validation SSIM')
        plt.title('SSIM')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(OUTPUT_DIR, 'training_ssim.png'))
        plt.show() 
        

# Training Execution

In [ ]:
if __name__ == "__main__":
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

    if not os.path.isdir(DATA_PATH):
        print(f"Error: Path not found {DATA_PATH}")
    else:
        train_path = os.path.join(DATA_PATH, "train")
        val_path = os.path.join(DATA_PATH, "validation")
        
        train_files = get_filenames(train_path)
        val_files = get_filenames(val_path)
        
        if train_files:
            print("\n Initializing Generators")
            
            train_gen = RamDataGenerator(
                train_path, train_files, batch_size=1, 
                patch_size=PATCH_SIZE, stride=PATCH_STRIDE, 
                shuffle=True, steps_per_epoch=STEPS_PER_EPOCH,
                augment=True, robust=True 
            )
            
            val_gen = RamDataGenerator(
                val_path, val_files, batch_size=1, 
                patch_size=PATCH_SIZE, stride=PATCH_STRIDE, 
                shuffle=False, augment=False, robust=False
            )
            model = build_har_cnn((None, None, IMG_CHANNELS))
            model.summary()

            if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
            
            save_best = ModelCheckpoint(os.path.join(OUTPUT_DIR, "har_cnn_v5_robust.keras"), save_best_only=True, monitor='val_loss', mode='min', verbose=1)
            stop_early = EarlyStopping(monitor='val_loss', patience=100, verbose=1, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1, min_lr=1e-6)
            csv_logger = CSVLogger(os.path.join(OUTPUT_DIR, "training_log.csv"), append=True)
            history = model.fit(
                train_gen,
                epochs=EPOCHS,
                steps_per_epoch=STEPS_PER_EPOCH,
                validation_data=val_gen,
                validation_steps=500, 
                callbacks=[save_best, stop_early, reduce_lr, csv_logger]
            )
            plot_history(history)